# Data Privacy
Differential Privacy (DP) adds noise to model updates to prevent
reconstructing individual data from gradients. This ensures that
removing or changing any single client's data has minimal impact
on the final model output.

**Key Concepts:**
- Adaptive Clipping: Bounds client contribution sensitivity
- Noise Multiplier: Controls privacy-utility tradeoff
- Client-side DP: Privacy applied before sending to server



In [1]:
# well flower framework seems to need the HF_TOKEN for evautaion but don't know why
import os
os.environ["HF_TOKEN"] = "hf_z**SHHSZ"
from huggingface_hub import whoami
# whoami()  # Confirms auth

In [2]:
import warnings
import logging
from logging import INFO, LogRecord, ERROR, DEBUG
from collections import OrderedDict

import torch 
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import Compose, Normalize, ToTensor

from flwr_datasets import FederatedDataset
from flwr.client import NumPyClient, Client, ClientApp
from flwr.client.mod import adaptiveclipping_mod
from flwr.common import ndarrays_to_parameters, Context
from flwr.common.logger import (
    ConsoleHandler,
    console_handler,
    FLOWER_LOGGER,
    LOG_COLORS,
)

from flwr.server import ServerApp, ServerConfig, ServerAppComponents
from flwr.server.strategy import (
    DifferentialPrivacyClientSideAdaptiveClipping,
    FedAvg,
)
from flwr.simulation import run_simulation

### Logging setup for Flower framework

In [3]:
class InfoFilter(logging.Filter):
    def filter(self, record):
        return record.levelno == ERROR

FLOWER_LOGGER.removeHandler(console_handler)
warnings.filterwarnings("ignore")

backend_setup = {"init_args": {"logging_level": ERROR, "log_to_driver": True}}

class ConsoleHandlerV2(ConsoleHandler):
    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)

    def format(self, record: LogRecord) -> str:
        """Format function that adds colors to log level."""
        if self.json:
            log_fmt = "{lvl='%(levelname)s', time='%(asctime)s', msg='%(message)s'}"
        else:
            log_fmt = (
                f"{LOG_COLORS[record.levelname] if self.colored else ''}"
                f"%(levelname)s {'%(asctime)s' if self.timestamps else ''}"
                f"{LOG_COLORS['RESET'] if self.colored else ''}"
                f": %(message)s"
            )
        formatter = logging.Formatter(log_fmt)
        return formatter.format(record)

console_handlerv2 = ConsoleHandlerV2(timestamps=False, json=False, colored=True)
console_handlerv2.setLevel(INFO)
console_handlerv2.addFilter(InfoFilter())
FLOWER_LOGGER.addHandler(console_handlerv2)

DEVICE = "cpu"

### Load the MNIST dataset
* Use `flwr-datasets` that provides with a Federated Dataset abstraction.

In [4]:
# Partition MNIST into 10 clients using flwr-datasets
def load_data(partition_id):
    fds = FederatedDataset(dataset="mnist", partitioners={"train": 10})
    partition = fds.load_partition(partition_id)

    traintest = partition.train_test_split(test_size=0.2, seed=42)
    traintest = traintest.with_transform(normalize)
    trainset, testset = traintest["train"], traintest["test"]

    trainloader = DataLoader(trainset, batch_size=64, shuffle=True)
    testloader = DataLoader(testset, batch_size=64)
    return trainloader, testloader

In [5]:
transforms = Compose([
    ToTensor(),
    Normalize((0.5,), (0.5,))
])

def normalize(batch):
    batch["image"] = [transforms(img) for img in batch["image"]]
    return batch

class SimpleModel(nn.Module):
    def __init__(self):
        super(SimpleModel, self).__init__()
        self.fc = nn.Linear(28*28, 128)
        self.relu = nn.ReLU()
        self.out = nn.Linear(128, 10)

    def forward(self, x):
        x =  torch.flatten(x, 1)
        x = self.fc(x)
        x = self.relu(x)
        x = self.out(x)
        return x

def train_model(net, trainloader, epochs: int=5):
    net.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(net.parameters())
    net.train()

    for _ in range(epochs):
        for batch in trainloader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(net(images), labels)
            loss.backward()
            optimizer.step()

def evaluate_model(net, testloader):
    net.to(DEVICE)
    criterion = torch.nn.CrossEntropyLoss()
    correct, loss = 0, 0.0
    with torch.no_grad():
        for batch in testloader:
            images = batch["image"].to(DEVICE)
            labels = batch["label"].to(DEVICE)
            outputs = net(images)
            loss += criterion(outputs, labels).item()
            correct += (
                (torch.max(outputs.data, 1)[1] == labels).sum().item()
            )
        accuracy = correct / len(testloader.dataset)
        return float(loss), float(accuracy)

def set_weights(net, parameters):
    params_dict = zip(net.state_dict().keys(), parameters)
    state_dict = OrderedDict(
        {k: torch.tensor(v) for k, v in params_dict}
    )
    net.load_state_dict(state_dict, strict=True)

def get_weights(net):
    ndarrays = [
        val.cpu().numpy() for _, val in net.state_dict().items()
    ]
    return ndarrays

### FlowerClient: trains on local data, returns updated weights to server


In [6]:
class FlowerClient(NumPyClient):
    def __init__(self, net, trainloader, testloader):
        self.net = net
        self.trainloader = trainloader
        self.testloader = testloader

    def fit(self, parameters, config):
        set_weights(self.net, parameters)
        train_model(self.net, self.trainloader)
        return get_weights(self.net), len(self.trainloader.dataset), {}

    def evaluate(self, parameters, config):
        set_weights(self.net, parameters)
        loss, accuracy = evaluate_model(self.net, self.testloader)
        return loss, len(self.testloader.dataset), {"accuracy": accuracy}


def client_fn(context: Context) -> Client:
    net = SimpleModel()
    partition_id = int(context.node_config["partition-id"])
    trainloader, testloader = load_data(partition_id=partition_id)
    return FlowerClient(net, trainloader, testloader).to_client()

### Define the ClientApp
Adaptive Clipping limits how much each client can influence
the model update, bounding the sensitivity of gradients.

This is a key mechanism in differential privacy.


In [7]:
client = ClientApp(
    client_fn,
    mods=[adaptiveclipping_mod],  # modifiers
)

### Define the Server side with the strategy FedAvg.

**DP:** Differential Privacy.

Server uses FedAvg + DifferentialPrivacyClientSideAdaptiveClipping.
Noise is added to aggregated updates, providing privacy guarantees
for individual client contributions.


In [8]:
net = SimpleModel()
params = ndarrays_to_parameters(get_weights(net))

def server_fn(context: Context):
    fedavg_without_dp = FedAvg(
        fraction_fit=0.6,
        fraction_evaluate=1.0,
        initial_parameters=params,
    )
    fedavg_with_dp = DifferentialPrivacyClientSideAdaptiveClipping(
        fedavg_without_dp,  # <- wrap the FedAvg strategy
        noise_multiplier=0.3,
        num_sampled_clients=6,
    )
    
    # Adjust to 50 rounds to ensure DP guarantees hold
    # with respect to the desired privacy budget
    config = ServerConfig(num_rounds=5)
    
    return ServerAppComponents(
        strategy=fedavg_with_dp,
        config=config,
    )

In [9]:
server = ServerApp(server_fn=server_fn)

### Run Client and Server apps.
Run FL with 10 clients, 60% participation per round, DP enabled

**Note**: This simulation may take approximately 7 to 10 minutes to complete all 50 rounds. 

In [10]:
run_simulation(server_app=server,
               client_app=client,
               num_supernodes=10,
               backend_config=backend_setup
)

(ClientAppActor pid=75322) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.1000.
(ClientAppActor pid=75322) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.0907. [repeated 2x across cluster] (Ray deduplicates logs by default. Set RAY_DEDUP_LOGS=0 to disable log deduplication, or see https://docs.ray.io/en/master/ray-observability/user-guides/configure-logging.html#log-deduplication for more options.)
(ClientAppActor pid=75321) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.0907. [repeated 5x across cluster]
(ClientAppActor pid=75325) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.0811.
(ClientAppActor pid=75321) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.0811.
(ClientAppActor pid=75325) INFO :      adaptiveclipping_mod: parameters are clipped by value: 0.0730. [repeated 5x across cluster]
(ClientAppActor pid=75325) INFO :      adaptiveclipping_mod: parameters are clipped by v